# Рекомендательная система для книжного сервиса

**Автор:** Куницын Денис

**Дата:** 1 сентября

## Цель и задачи

**Бизнес-задача:** книжный сервис ReadMe с каталогом около пятисот книг показывает всем пользователям на главной странице один и тот же блок «Популярное». Такие рекомендации не учитывают интересы читателей — любители нон-фикшена видят самое популярное фэнтези и не идут его читать, — поэтому конверсия с главной страницы в чтение почти не растёт.

**Цель в терминах Machine Learning:** собрать пайплайн рекомендательной системы и обоснованно выбрать модель на замену популярному списку. Главный вопрос проекта: **какая модель рекомендаций работает лучше на небольшом каталоге и почему?** Каталог мал, поэтому двухуровневая архитектура с генерацией кандидатов здесь избыточна: скоринг всех книг считается за секунды.

**Условия:**

 * список топ-K рекомендаций при `K = 10` для пользователей с тёплым стартом;
 * релевантной считается книга, которой пользователь в тесте поставил оценку не ниже 4;
 * модели — popularity baseline, content-based, item-based CF и ALS;
 * метрики — precision@10, recall@10, MAP@10, NDCG@10 и coverage.

**Задачи:**

1. Загрузить данные, проверить типы, пропуски, дубликаты и корректность ключей.
2. Провести EDA: масштаб данных, заполненность матрицы взаимодействий, распределения событий и оценок.
3. Разделить события по времени на train, validation и test.
4. Реализовать офлайн-метрики и единый формат предсказаний.
5. Построить и оценить popularity baseline, content-based, item-based CF и ALS.
6. Сравнить модели и выбрать решение для продакшена.

## Данные

- `interactions.csv` — история взаимодействий: `user_id`, `item_id`, `rating` (1–5), `timestamp`;
- `items.csv` — каталог книг: `item_id`, `title`, `author`, `description`, `category`, `year`;
- `users.csv` — читатели: `user_id`, `age_group`, `city`.

> Папка с данными задаётся переменной окружения `DATA_DIR` (по умолчанию — `datasets`).

## Подготовка к работе: установка библиотек и настройка окружения

Подключаем библиотеки, задаём константы проекта и фиксируем сиды `random` и `numpy`.

In [1]:
import os

# Ограничиваем внутренние потоки BLAS: ALS из библиотеки implicit работает на CPU стабильнее и без предупреждений
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder, normalize
import implicit

from IPython.display import display

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RANDOM_STATE = 42   # Сид воспроизводимости
K = 10              # Длина списка рекомендаций
REL_THRESHOLD = 4   # Порог релевантности: оценка >= 4
CUTOFF_VAL_Q = 0.70   # Временной порог разделения train и validation
CUTOFF_TEST_Q = 0.85  # Временной порог разделения validation и test

DATA_DIR = Path(os.getenv("DATA_DIR", "datasets"))

def set_seed(seed: int = RANDOM_STATE) -> None:
    # Фиксируем сиды random и numpy
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_STATE)
print("Сид зафиксирован:", RANDOM_STATE)


Сид зафиксирован: 42


## Шаг 1. Загрузка данных и первичный осмотр

Три таблицы проверяются на типичные проблемы: типы данных, пропуски, дубликаты, корректность ключей.

In [3]:
interactions = pd.read_csv(DATA_DIR / "interactions.csv", parse_dates=["timestamp"])
items = pd.read_csv(DATA_DIR / "items.csv")
users = pd.read_csv(DATA_DIR / "users.csv")

print("interactions:", interactions.shape, "| items:", items.shape, "| users:", users.shape)
display(interactions.head())
display(items.head(3))
display(users.head(3))

interactions: (26240, 4) | items: (500, 6) | users: (1500, 3)


,user_id,item_id,rating,timestamp
0,268,61,4,2023-01-04 01:43:50
1,1240,219,5,2023-01-04 04:55:00
2,833,362,3,2023-01-04 13:14:37
3,1240,185,5,2023-01-04 23:22:12
4,268,447,4,2023-01-05 04:46:31


,item_id,title,author,description,category,year
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,"История о поиске себя, в которой каждый узнает...",янг-эдалт,2008.0
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling,"История о избранном герое, которому предстоит ...",фэнтези,1997.0
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,"История о избранном герое, которому предстоит ...",фэнтези,NaN


,user_id,age_group,city
0,1,18–25,Новосибирск
1,2,36–50,Казань
2,3,50+,Москва


In [4]:
# Собираем сводку по каждой таблице: типы колонок, число пропусков, число уникальных значений
for name, table in [("interactions", interactions), ("items", items), ("users", users)]:
    print(f"--- {name} ---")
    display(pd.DataFrame({
        "dtype": table.dtypes.astype(str),
        "missing": table.isna().sum(),
        "unique": table.nunique(),
    }))

--- interactions ---


,dtype,missing,unique
user_id,int64,0,1500
item_id,int64,0,500
rating,int64,0,5
timestamp,datetime64[us],0,26202


--- items ---


,dtype,missing,unique
item_id,int64,0,500
title,str,0,500
author,str,0,317
description,str,0,301
category,str,0,7
year,float64,20,124


--- users ---


,dtype,missing,unique
user_id,int64,0,1500
age_group,str,0,4
city,str,0,6


In [5]:
# Находим полные дубликаты строк в interactions и удаляем их
n_full_duplicates = int(interactions.duplicated().sum())
print("Полных дубликатов строк:", n_full_duplicates)

interactions = interactions.drop_duplicates().copy()
print("После удаления:", interactions.shape)

# Проверяем, остались ли повторные пары «пользователь — айтем» (без учёта времени)
print("Повторных пар user-item:", int(interactions.duplicated(["user_id", "item_id"]).sum()))

Полных дубликатов строк: 30
После удаления: (26210, 4)
Повторных пар user-item: 0


In [6]:
# Проверяем корректность ключей
# Нужно узнать, сколько событий ссылаются на item_id, которого нет в items, и на user_id, которого нет в users
print("Событий с неизвестным item_id:", int((~interactions["item_id"].isin(items["item_id"])).sum()))
print("Событий с неизвестным user_id:", int((~interactions["user_id"].isin(users["user_id"])).sum()))

Событий с неизвестным item_id: 0
Событий с неизвестным user_id: 0


In [7]:
# Считаем пропуски в поле year таблицы items и принимаем решение, что с ними делать
# Заполняем пропуски медианой по каталогу
print("Пропусков в year:", int(items["year"].isna().sum()))

year_median = items["year"].median()
items["year"] = items["year"].fillna(year_median)
print(f"Пропуски заполнены медианой: {year_median:.0f}")

Пропусков в year: 20
Пропуски заполнены медианой: 1998


**Выводы по шагу 1.**

В interactions найдено 30 полных дубликатов; они не добавляют новых взаимодействий, поэтому удалены. После очистки осталось 26 210 строк, повторных пар "пользователь - айтем" нет.

Событий с неизвестными item_id и user_id не обнаружено, поэтому все взаимодействия можно корректно связать со справочниками до построения моделей. В items было 20 пропусков в year; книги не исключались из небольшого каталога, а пропуски заполнены медианой 1998.

## Шаг 2. EDA

Свойства данных, от которых зависит моделирование: масштаб, заполненность матрицы взаимодействий, распределения событий и оценок.

In [8]:
# Оцениваем размер данных: число пользователей, айтемов, взаимодействий
# Рассчитываем заполненность матрицы взаимодействий (количество событий, делённое на количество возможных пар «пользователь — айтем»)
n_users = interactions["user_id"].nunique()
n_items = interactions["item_id"].nunique()
n_events = len(interactions)
density = n_events / (n_users * n_items)

print("Пользователей:", n_users, "| Айтемов:", n_items, "| Взаимодействий:", n_events)
print(f"Заполненность матрицы взаимодействий: {density:.2%}")

Пользователей: 1500 | Айтемов: 500 | Взаимодействий: 26210
Заполненность матрицы взаимодействий: 3.49%


In [9]:
# Проверяем распределения числа событий по пользователям и по айтемам
per_user = interactions.groupby("user_id").size()
per_item = interactions.groupby("item_id").size()

print("События на пользователя:")
print(per_user.describe().round(1))
print()
print("События на айтем:")
print(per_item.describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(per_user, bins=17)
axes[0].set_title("События на пользователя")
axes[0].set_xlabel("Число событий")
axes[0].set_ylabel("Пользователей")
axes[1].hist(per_item, bins=30)
axes[1].set_title("События на айтем")
axes[1].set_xlabel("Число событий")
axes[1].set_ylabel("Айтемов")
plt.tight_layout()
plt.show()

События на пользователя:
count    1500.0
mean       17.5
std         4.7
min         8.0
25%        14.0
50%        18.0
75%        22.0
max        24.0
dtype: float64

События на айтем:
count    500.0
mean      52.4
std       35.4
min        9.0
25%       29.0
50%       40.0
75%       65.2
max      258.0
dtype: float64


/var/folders/d2/9r9ntqz94pb1bt824sbqy1w80000gn/T/ipykernel_97630/3532998359.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Оцениваем:
# - какая доля айтемов собрала меньше 10 событий;
# - какая доля всех событий приходится на верхние 10% айтемов
share_items_lt10 = (per_item < 10).mean()
top10_items = per_item.nlargest(max(1, int(np.ceil(len(per_item) * 0.10))))
share_top10_events = top10_items.sum() / n_events

print(f"Айтемов с менее чем 10 событиями: {share_items_lt10:.1%}")
print(f"Доля событий у топ-10% айтемов: {share_top10_events:.1%}")

Айтемов с менее чем 10 событиями: 0.2%
Доля событий у топ-10% айтемов: 25.5%


In [11]:
# Считаем распределение оценок (доли значений от 1 до 5)
# Рассчитываем долю оценок не ниже порога релевантности REL_THRESHOLD
rating_distribution = interactions["rating"].value_counts(normalize=True).sort_index().reindex(range(1, 6), fill_value=0)
display(rating_distribution.round(3).rename("share").to_frame())

share_high = (interactions["rating"] >= REL_THRESHOLD).mean()
print(f"Доля оценок >= {REL_THRESHOLD}: {share_high:.1%}")

plt.figure(figsize=(5, 3))
plt.bar(rating_distribution.index.astype(str), rating_distribution.values)
plt.title("Распределение оценок")
plt.xlabel("Оценка")
plt.ylabel("Доля событий")
plt.show()

,share
rating,
1,0.031
2,0.071
3,0.244
4,0.324
5,0.331


Доля оценок >= 4: 65.5%


/var/folders/d2/9r9ntqz94pb1bt824sbqy1w80000gn/T/ipykernel_97630/2920827737.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Выводы по шагу 2.**

После удаления дубликатов в данных 1 500 пользователей, 500 книг и 26 210 взаимодействий. Матрица заполнена лишь на 3,49%, то есть большинство пар "пользователь - книга" не наблюдалось.

Распределение по книгам неравномерно: у верхних 10% книг сосредоточено 25,5% событий, тогда как 0,2% книг имеют менее 10 событий. Поэтому популярность - сильный глобальный сигнал, а у моделей, использующих поведение, данных для менее популярных книг существенно меньше. Оценка - эксплицитный фидбэк: релевантность в тесте определена непосредственно выставленной оценкой rating >= 4; таких событий 65,5%.

## Шаг 3. Временной сплит

События делятся по времени: обучение происходит на самых ранних данных, валидация и проверка — на более поздних. Два порога-квантиля делят события на три части: `train` — для обучения всех моделей, `validation` — для подбора гиперпараметров ALS, `test` — для финального тестирования.

Каталог, история пользователей и вся статистика дальше строятся только по `train`.

In [12]:
# Разделяем события по времени на три части: пороги-квантили CUTOFF_VAL_Q и CUTOFF_TEST_Q по timestamp
# train — строго до первого порога
# validation — между порогами
# test — от второго порога и позже
cutoff_val = interactions["timestamp"].quantile(CUTOFF_VAL_Q)
cutoff_test = interactions["timestamp"].quantile(CUTOFF_TEST_Q)
train = interactions[interactions["timestamp"] < cutoff_val].copy()
validation = interactions[(interactions["timestamp"] >= cutoff_val) & (interactions["timestamp"] < cutoff_test)].copy()
test = interactions[interactions["timestamp"] >= cutoff_test].copy()

print("Порог train|validation:", cutoff_val)
print("Порог validation|test:", cutoff_test)
print("train:", len(train), "| validation:", len(validation), "| test:", len(test))


Порог train|validation: 2024-04-12 04:34:05.500000
Порог validation|test: 2024-07-06 23:35:54.350000
train: 18347 | validation: 3931 | test: 3932


In [13]:
# Собираем по train отсортированный список айтемов каталога item_ids
# Собираем по train словарь id_to_row (item_id -> номер строки в матрицах)
# Собираем по train множество catalog и словарь train_by_user (user_id -> список айтемов его train-истории)
item_ids = sorted(train["item_id"].unique())
id_to_row = {item_id: row for row, item_id in enumerate(item_ids)}
catalog = set(item_ids)
train_by_user = train.groupby("user_id")["item_id"].agg(list).to_dict()

print("Каталог train:", len(item_ids), "айтемов")
print("Пользователей с train-историей:", len(train_by_user))

Каталог train: 500 айтемов
Пользователей с train-историей: 1343


In [14]:
# Оставляем в validation и test только пользователей с тёплым стартом — тех, у кого есть train-история
validation = validation[validation["user_id"].isin(train_by_user)].copy()
test = test[test["user_id"].isin(train_by_user)].copy()
test_users = sorted(test["user_id"].unique())

print("validation после warm-фильтрации:", len(validation))
print("test после warm-фильтрации:", len(test), "| warm-пользователей:", len(test_users))


validation после warm-фильтрации: 3188
test после warm-фильтрации: 1959 | warm-пользователей: 392


In [15]:
# Проверки честности разделения: все проверки должны проходить без ошибок
assert train["timestamp"].max() < validation["timestamp"].min(), "train охватывает validation"
assert validation["timestamp"].max() < test["timestamp"].min(), "validation охватывает test"

overlap = pd.concat([
    train.merge(validation, on=["user_id", "item_id", "timestamp"]),
    train.merge(test, on=["user_id", "item_id", "timestamp"]),
    validation.merge(test, on=["user_id", "item_id", "timestamp"]),
])
assert overlap.empty, "Части выборок пересекаются по событиям"

print("Разделение честное: каждая выборка целиком в прошлом относительно следующей, пересечений нет.")


Разделение честное: каждая выборка целиком в прошлом относительно следующей, пересечений нет.


**Выводы по шагу 3.**

Сплит сделан по времени: модель обучается на прошлом и проверяется на последующих событиях, поэтому не использует будущую информацию. Каталог из 500 книг и истории пользователей сформированы только по train; это исключает попадание в рекомендации книг и взаимодействий, которые на момент обучения ещё не были известны.

Два assert проверяют, что максимальное время каждой части раньше минимального времени следующей, а также что между частями нет одинаковых событий. Validation выделена отдельно для выбора числа факторов ALS: в ней 600 пользователей с непустым ground truth, а test оставлен для единственной финальной оценки.

## Шаг 4. Офлайн-метрики и единый формат предсказаний

Все модели проекта оцениваются одной функцией. Рекомендатель — это функция `recommender(user_id, n)`, которая возвращает список пар `(item_id, score)` по убыванию `score`; предсказания хранятся в едином формате `user_id`, `item_id`, `rank`, `score`, `model_id`, где `rank` начинается с 1.

In [16]:
def precision_at_k(recs, rel, k):
    """Доля релевантных айтемов среди первых k рекомендаций"""
    # Считаем, сколько из первых k рекомендаций попали в rel, и делим на k
    if k <= 0 or not recs:
        return 0.0
    return sum(item_id in rel for item_id, _ in recs[:k]) / k


def recall_at_k(recs, rel, k):
    """Доля найденных релевантных от всех релевантных айтемов пользователя"""
    # Числитель тот же, что в precision, а знаменатель — len(rel)
    if not rel:
        return 0.0
    return sum(item_id in rel for item_id, _ in recs[:k]) / len(rel)


def map_at_k(recs, rel, k):
    """Average precision: точность в точках попаданий, усреднённая по релевантным"""
    # Идём по рекомендациям с позициями от 1: на каждом попадании прибавляем «(число попаданий к этому моменту) / позиция»
    # Сумму делим на min(len(rel), k)
    if not rel or k <= 0:
        return 0.0
    hits = 0
    score = 0.0
    for pos, (item_id, _) in enumerate(recs[:k], start=1):
        if item_id in rel:
            hits += 1
            score += hits / pos
    return score / min(len(rel), k)


def ndcg_at_k(recs, rel, k):
    """NDCG с бинарной релевантностью: DCG = сумма 1 / log2(позиция + 1)"""
    # DCG: каждый релевантный айтем на позиции pos добавляет 1 / log2(pos + 1)
    # IDCG — тот же DCG для идеального порядка (все релевантные сверху, но не больше k)
    # Возвращаем DCG / IDCG, аккуратно обработав IDCG = 0
    if not rel or k <= 0:
        return 0.0
    dcg = sum(1 / np.log2(pos + 1) for pos, (item_id, _) in enumerate(recs[:k], start=1) if item_id in rel)
    ideal_len = min(len(rel), k)
    idcg = sum(1 / np.log2(pos + 1) for pos in range(1, ideal_len + 1))
    return dcg / idcg if idcg else 0.0

In [17]:
# Быстрая проверка метрик на пробном примере:
# Рекомендованы айтемы [1, 2, 3], релевантны {2, 4}
toy_recs = [(1, 0.9), (2, 0.8), (3, 0.7)]
toy_rel = {2, 4}

assert precision_at_k(toy_recs, toy_rel, 3) == 1 / 3
assert recall_at_k(toy_recs, toy_rel, 3) == 1 / 2
assert map_at_k(toy_recs, toy_rel, 3) == (1 / 2) / 2
assert abs(ndcg_at_k(toy_recs, toy_rel, 3) - (1 / np.log2(3)) / (1 + 1 / np.log2(3))) < 1e-9
print("Метрики на пробном примере посчитаны верно")

Метрики на пробном примере посчитаны верно


In [18]:
def positives_by_user(part):
    """user_id -> множество айтемов части данных с rating >= REL_THRESHOLD"""
    # Отбираем события с высокой оценкой и группируем айтемы по пользователям
    positive = part[part["rating"] >= REL_THRESHOLD]
    return positive.groupby("user_id")["item_id"].agg(set).to_dict()


pos_by_user = positives_by_user(test)
pos_by_user_val = positives_by_user(validation)


def make_ground_truth(positives):
    """Релевантные айтемы пользователя: rating >= порога, в каталоге train,
    не из его train-истории"""

    def ground_truth(user_id):
        # Берём позитивы пользователя из positives и оставляем только те,
        # что есть в каталоге train и которых нет в его train-истории
        return (set(positives.get(user_id, set())) & catalog) - set(train_by_user.get(user_id, []))

    return ground_truth


ground_truth = make_ground_truth(pos_by_user)          # Финальная оценка на test
ground_truth_val = make_ground_truth(pos_by_user_val)  # Подбор гиперпараметров ALS

# Оцениваем только пользователей с непустым ground truth
eval_users = [user_id for user_id in test_users if ground_truth(user_id)]
val_users = [user_id for user_id in validation["user_id"].unique() if ground_truth_val(user_id)]
print("Пользователей в оценке (eval_users):", len(eval_users))
print("Пользователей в валидации (val_users):", len(val_users))


Пользователей в оценке (eval_users): 354
Пользователей в валидации (val_users): 600


In [19]:
def evaluate(recommender, users, k=K, truth=None):
    """Метрики по каждому пользователю с макро-усреднением + coverage"""
    truth = truth or ground_truth  # по умолчанию оцениваем на test
    acc = {name: [] for name in ["Precision@K", "Recall@K", "MAP@K", "NDCG@K"]}
    recommended_items = set()
    for user_id in users:
        rel = truth(user_id)
        recs = recommender(user_id, k)
        recommended_items.update(item_id for item_id, _ in recs)
        acc["Precision@K"].append(precision_at_k(recs, rel, k))
        acc["Recall@K"].append(recall_at_k(recs, rel, k))
        acc["MAP@K"].append(map_at_k(recs, rel, k))
        acc["NDCG@K"].append(ndcg_at_k(recs, rel, k))
    result = {name: float(np.mean(values)) for name, values in acc.items()}
    result["Coverage"] = len(recommended_items) / len(item_ids)  # доля каталога в выдаче
    return result


def predictions_frame(recommender, users, model_id, n=K):
    """Единый формат предсказаний: одна строка — одна рекомендация"""
    rows = []
    for user_id in users:
        for rank, (item_id, score) in enumerate(recommender(user_id, n), start=1):
            rows.append((user_id, item_id, rank, score, model_id))
    return pd.DataFrame(rows, columns=["user_id", "item_id", "rank", "score", "model_id"])


**Выводы по шагу 4.**

Precision@10 измеряет долю релевантных книг среди десяти показанных, Recall@10 - долю всех релевантных книг пользователя, найденных в десятке, MAP@10 и NDCG@10 учитывают позиции релевантных книг, а coverage - долю каталога, которая появилась хотя бы в одной выдаче. Coverage дополняет метрики точности: она показывает, насколько модели задействуют каталог, а не только попадают в известные популярные книги.

Метрики усредняются макро-способом, поэтому вклад каждого пользователя одинаков. В eval_users вошли 354 пользователя тёплого старта с непустым набором релевантных книг в test; единый пул нужен, чтобы различия метрик отражали модели, а не различия в проверяемых пользователях.

## Шаг 5. Popularity baseline

Отправная точка сравнения — то, что сервис делает сейчас: рекомендует всем популярное. Популярность считается по числу взаимодействий и только на train.

In [20]:
# Популярность по числу взаимодействий — только на train
pop = train["item_id"].value_counts()
top_pop = pop.index.tolist()

# Топ-5 популярных книг: это сейчас видят все пользователи сервиса
top5 = items.set_index("item_id").loc[pop.head(5).index, ["title", "category"]]
top5["events"] = pop.head(5).values
display(top5)


def recommend_popular(user_id, n=K):
    """Top-N популярных айтемов без тех, что пользователь уже видел в train"""
    # Идём по top_pop сверху вниз, пропускаем айтемы из train-истории пользователя
    # Собираем пары (item_id, score), где score — счётчик популярности
    # Остановитесь, когда наберёте n рекомендаций.
    seen = set(train_by_user.get(user_id, []))
    recs = []
    for item_id in top_pop:
        if item_id in seen:
            continue
        recs.append((item_id, float(pop[item_id])))
        if len(recs) >= n:
            break
    return recs

,title,category,events
item_id,,,
1,"The Hunger Games (The Hunger Games, #1)",янг-эдалт,187
61,The Girl on the Train,детектив,171
2,Harry Potter and the Sorcerer's Stone (Harry P...,фэнтези,152
3,"Twilight (Twilight, #1)",фэнтези,149
17,"Catching Fire (The Hunger Games, #2)",янг-эдалт,147


In [21]:
pred_pop = predictions_frame(recommend_popular, eval_users, "popularity")
display(pred_pop.head())

metrics_pop = evaluate(recommend_popular, eval_users)
print({name: round(value, 4) for name, value in metrics_pop.items()})

,user_id,item_id,rank,score,model_id
0,3,1,1,187.0,popularity
1,3,61,2,171.0,popularity
2,3,2,3,152.0,popularity
3,3,3,4,149.0,popularity
4,3,17,5,147.0,popularity


{'Precision@K': 0.0282, 'Recall@K': 0.0653, 'MAP@K': 0.0213, 'NDCG@K': 0.0463, 'Coverage': 0.028}


**Выводы по шагу 5.**

Popularity baseline получил Precision@10 = 0,0282, Recall@10 = 0,0653, MAP@10 = 0,0213 и NDCG@10 = 0,0463. Модель даёт некоторую точность за счёт глобальной популярности, но не учитывает различия между интересами пользователей.

Coverage равен 0,028: в выдачах использовано только 2,8% каталога. Это согласуется с проблемой исходного блока "Популярное": пользователи видят очень узкий общий набор книг.

## Шаг 6. Модель content-based

Модель рекомендует книги, похожие на те, что пользователю уже понравились, по признакам книг из каталога. Айтем-профили собираются из трёх типов признаков:

 * категорий, закодированных One-Hot Encoding;
 * объединённых названий и описаний книг, векторизованных с помощью TF-IDF;
 * года издания книги, нормализованного к диапазону [0, 1].

Профиль пользователя строится только по книгам, которые получили в train оценку 4–5. Низкие оценки означают, что книга читателю не понравилась, и формировать по ним профиль ошибочно: модель стала бы советовать заведомо неудачные книги. Профили отобранных книг взвешиваются оценкой, суммируются и L2-нормируются; рекомендации — топ-10 по близости.

In [22]:
# Берём признаки из каталога, но строки матрицы — в порядке item_ids (каталог train)
items_train = items.set_index("item_id").reindex(item_ids)

# Кодируем категорию через OneHotEncoder
# Результат — разреженная матрица C
category_encoder = OneHotEncoder()
C = category_encoder.fit_transform(items_train[["category"]])

# Собираем текст «название + описание»
# Кодируем текст через TfidfVectorizer(max_features=300) в матрицу T
text = items_train["title"].fillna("") + " " + items_train["description"].fillna("")
T = TfidfVectorizer(max_features=300).fit_transform(text)

# Нормируем год к диапазону [0, 1] (матрица Y из одного столбца)
year = items_train["year"].values.reshape(-1, 1)
Y = (year - year.min()) / (year.max() - year.min())

# Склеиваем блоки с весами 1.0 / 1.0 / 0.5 (hstack) и L2-нормируем строки (normalize)
item_mat = normalize(hstack([C, T, 0.5 * csr_matrix(Y)]), norm="l2", axis=1)
print("матрица айтем-профилей:", item_mat.shape)

матрица айтем-профилей: (500, 308)


In [23]:
def build_user_profiles(train_df):
    """Профиль интересов: взвешенная по оценке сумма айтем-профилей книг
    с rating >= REL_THRESHOLD из train-истории, L2-нормировка"""
    # Оставляем только события с rating >= REL_THRESHOLD: интерес выражают высокие оценки
    positive = train_df[train_df["rating"] >= REL_THRESHOLD]
    profiles = {}
    for user_id, group in positive.groupby("user_id"):
        # Собираем номера строк книг пользователя (id_to_row)
        # Берём его оценки как веса
        # Складываем взвешенные айтем-профили из item_mat в один вектор
        # L2-нормируем полученный вектор
        # Пользователей с нулевым вектором пропускаем
        rows = [id_to_row[item_id] for item_id in group["item_id"] if item_id in id_to_row]
        weights = group.loc[group["item_id"].isin(id_to_row), "rating"].to_numpy(dtype=float)
        profile = np.asarray(item_mat[rows].multiply(weights.reshape(-1, 1)).sum(axis=0)).ravel()
        profile_norm = np.linalg.norm(profile)
        if profile_norm > 0:
            profiles[user_id] = profile / profile_norm
    return profiles


user_profiles = build_user_profiles(train)
print("Профилей пользователей:", len(user_profiles))


Профилей пользователей: 1319


In [24]:
def recommend_content(user_id, n=K):
    """Top-N айтемов по близости к профилю интересов, без уже виденного"""
    if user_id not in user_profiles:
        return []
    scores = item_mat.dot(user_profiles[user_id])  # Косинус после L2-нормировки
    seen = set(train_by_user.get(user_id, []))
    recs = []
    for row in np.argsort(-scores):
        item_id = item_ids[row]
        if item_id in seen:
            continue
        recs.append((item_id, float(scores[row])))
        if len(recs) >= n:
            break
    return recs


pred_content = predictions_frame(recommend_content, eval_users, "content_based")
metrics_content = evaluate(recommend_content, eval_users)
print({name: round(value, 4) for name, value in metrics_content.items()})

{'Precision@K': 0.0201, 'Recall@K': 0.0557, 'MAP@K': 0.0185, 'NDCG@K': 0.0394, 'Coverage': 0.966}


**Выводы по шагу 6.**

Content-based модель персонализирует выдачу по близости профиля пользователя, построенного из высоко оценённых им книг, к признакам книг: категории, объединённому тексту названия и описания и году издания.

По всем четырём метрикам точности она уступила popularity baseline: Precision@10 0,0201 против 0,0282, Recall@10 0,0557 против 0,0653, MAP@10 0,0185 против 0,0213 и NDCG@10 0,0394 против 0,0463. Зато coverage вырос с 0,028 до 0,966: модель использует почти весь каталог, но признаки содержания сами по себе оказались слабее глобальной популярности для ранжирования релевантных книг.

## Шаг 7. Item-based CF

Коллаборативная фильтрация не смотрит на признаки книг — только на поведение пользователей. Похожими считаются айтемы, которые высоко оценивают одни и те же читатели.

Используются положительные взаимодействия: оценка 4–5 в train — сигнал интереса, которому соответствует единица в матрице. По разреженной бинарной матрице user × item считается косинусное сходство книг; скор кандидата — сумма его сходств с книгами, получившими высокую оценку пользователя в train.

In [25]:
# Оставляем положительные взаимодействия train (rating >= REL_THRESHOLD)
# Строим по ним разреженную бинарную матрицу user x item
# На строках матрицы — пользователи (в порядке user_list)
# В столбцах матрицы — айтемы (в порядке item_ids)
# Значения матрицы — единицы
train_positive = train[train["rating"] >= REL_THRESHOLD].drop_duplicates(["user_id", "item_id"])

user_list = sorted(train_by_user)
user_to_row = {user_id: row for row, user_id in enumerate(user_list)}

R = csr_matrix((np.ones(len(train_positive), dtype=np.float32), (train_positive["user_id"].map(user_to_row), train_positive["item_id"].map(id_to_row))), shape=(len(user_list), len(item_ids)))
print("Матрица положительных взаимодействий:", R.shape, "| Ненулевых значений:", R.nnz)

# Считаем косинусное сходство айтемов по столбцам матрицы
# Обнуляем диагональ — похожесть айтема на самого себя не учитываем
item_sim = cosine_similarity(R.T, dense_output=False)
item_sim.setdiag(0)
item_sim.eliminate_zeros()

print("Матрица похожести айтемов:", item_sim.shape)


Матрица положительных взаимодействий: (1343, 500) | Ненулевых значений: 11868
Матрица похожести айтемов: (500, 500)


In [26]:
def recommend_item_cf(user_id, n=K):
    """Скор кандидата — сумма его сходств с книгами положительной train-истории"""
    if user_id not in user_to_row:
        return []
    # Берём вектор пользователя (строка R; единицы у книг с высокой оценкой)
    # Для пользователя без положительной истории возвращаем пустой список
    # Считаем скор всех айтемов как item_sim @ вектор
    # Собираем топ-K без уже виденного — по образцу recommend_content
    user_vector = R.getrow(user_to_row[user_id])
    if user_vector.nnz == 0:
        return []
    scores = (item_sim @ user_vector.T).toarray().ravel()
    seen = set(train_by_user.get(user_id, []))
    recs = []
    for row in np.argsort(-scores):
        item_id = item_ids[row]
        if item_id in seen:
            continue
        recs.append((item_id, float(scores[row])))
        if len(recs) >= n:
            break
    return recs


pred_item_cf = predictions_frame(recommend_item_cf, eval_users, "item_based_cf")
metrics_item_cf = evaluate(recommend_item_cf, eval_users)
print({name: round(value, 4) for name, value in metrics_item_cf.items()})


{'Precision@K': 0.1008, 'Recall@K': 0.2769, 'MAP@K': 0.1446, 'NDCG@K': 0.2208, 'Coverage': 0.88}


**Выводы по шагу 7.**

Item-based CF превзошла обе предыдущие модели по всем метрикам точности: NDCG@10 равен 0,2208 против 0,0463 у popularity и 0,0394 у content-based; Precision@10 - 0,1008, Recall@10 - 0,2769, MAP@10 - 0,1446. Это показывает ценность поведенческого сигнала: модель находит книги, которые высоко оценивали читатели, связанные через уже понравившиеся пользователю книги.

Скор кандидата - сумма косинусных сходств кандидата с книгами из положительной train-истории пользователя. Уже виденные книги исключаются, чтобы не рекомендовать пользователю известные ему позиции и чтобы оценка отражала поиск новых книг.

## Шаг 8. ALS

Для матричной факторизации используется оптимизированная библиотека `implicit`.

Имплицитный фидбэк — бинаризованные оценки (`rating >= REL_THRESHOLD` → 1), то есть бинарная матрица положительных взаимодействий из шага 7. Формулу `confidence = 1 + alpha * значение` библиотека строит сама.

Рекомендации формируются без уже знакомых книг, а число факторов подбирается мини-экспериментом по NDCG@10 на валидационной выборке.

In [27]:
ALS_FACTORS = 20
ALS_REGULARIZATION = 0.05
ALS_ITERATIONS = 15  # Итерации библиотеки вычислительно дешёвые
ALS_ALPHA = 40

# Матрица для ALS уже готова
# Бинарная матрица R из шага 7 — это implicit feedback после бинаризации оценок
print("положительных взаимодействий в train:", R.nnz)


положительных взаимодействий в train: 11868


In [28]:
# Обучаем implicit.als.AlternatingLeastSquares на матрице R
# Гиперпараметры — константы выше, random_state=RANDOM_STATE, обучение без прогресс-бара
# alpha передаётся модели, confidence = 1 + alpha * значение она строит сама
als_model = implicit.als.AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REGULARIZATION,
    iterations=ALS_ITERATIONS,
    alpha=ALS_ALPHA,
    random_state=RANDOM_STATE,
)
als_model.fit(R, show_progress=False)

print("Факторы пользователей:", als_model.user_factors.shape)
print("Факторы айтемов:", als_model.item_factors.shape)


Факторы пользователей: (1343, 20)
Факторы айтемов: (500, 20)


In [29]:
def make_als_recommender(model):
    """Рекомендатель по факторам модели: score — скалярное произведение факторов"""

    def recommend(user_id, n=K):
        if user_id not in user_to_row:
            return []
        scores = model.user_factors[user_to_row[user_id]] @ model.item_factors.T
        seen = set(train_by_user.get(user_id, []))
        recs = []
        for row in np.argsort(-scores):
            item_id = item_ids[row]
            if item_id in seen:
                continue
            recs.append((item_id, float(scores[row])))
            if len(recs) >= n:
                break
        return recs

    return recommend


# Пример: первые три рекомендации базовой модели для одного пользователя
print(make_als_recommender(als_model)(eval_users[0])[:3])


[(np.int64(92), 0.8285898566246033), (np.int64(454), 0.8136481046676636), (np.int64(15), 0.7800930738449097)]


In [30]:
# Мини-эксперимент: число факторов подбираем по NDCG@10 на validation
factor_grid = [20, 64, 128]

grid_rows = []
grid_models = {}
for factors in factor_grid:
    grid_model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=ALS_REGULARIZATION,
        iterations=ALS_ITERATIONS,
        alpha=ALS_ALPHA,
        random_state=RANDOM_STATE,
    )
    grid_model.fit(R, show_progress=False)
    grid_models[factors] = grid_model
    # Оцениваем конфигурацию на валидации: пул val_users и ground_truth_val
    val_metrics = evaluate(make_als_recommender(grid_model), val_users, truth=ground_truth_val)
    grid_rows.append({"factors": factors, "NDCG@10 (validation)": round(val_metrics["NDCG@K"], 4)})

grid_table = pd.DataFrame(grid_rows).set_index("factors")
display(grid_table)

# Выбираем лучшее число факторов на данных validation
best_factors = int(grid_table["NDCG@10 (validation)"].idxmax())
print("лучшее число факторов по validation:", best_factors)

# Финальная ALS — лучшая на данных по validation
# Её метрики на test считаем один раз
recommend_als = make_als_recommender(grid_models[best_factors])
pred_als = predictions_frame(recommend_als, eval_users, "als")
metrics_als = evaluate(recommend_als, eval_users)
print({name: round(value, 4) for name, value in metrics_als.items()})


,NDCG@10 (validation)
factors,
20,0.1754
64,0.1982
128,0.1739


лучшее число факторов по validation: 64
{'Precision@K': 0.0791, 'Recall@K': 0.2214, 'MAP@K': 0.1239, 'NDCG@K': 0.1865, 'Coverage': 0.962}


**Выводы по шагу 8.**

ALS обучается на имплицитном сигнале: оценки rating >= 4 преобразованы в единицы положительных взаимодействий. Бинаризация оставляет признак интереса, а alpha = 40 задаёт силу confidence для этих взаимодействий в модели.

Лучшее значение на validation получено при 64 факторах: NDCG@10 = 0,1982; при 20 факторах - 0,1754, при 128 - 0,1739. Поэтому выбран вариант с 64 факторами; большее число факторов здесь не улучшило качество. На test ALS лучше baseline по Precision@10, Recall@10, MAP@10 и NDCG@10 (0,1865 против 0,0463 по NDCG@10), но уступает item-based CF по этим метрикам; число факторов подбиралось на validation, чтобы test остался независимой финальной проверкой.

## Шаг 9. Сравнение моделей и выводы

Метрики всех моделей сводятся в одну таблицу. Сравнение корректно: у всех моделей одно разделение на выборки, один `K`, одинаковое исключение виденного и один пул пользователей `eval_users`.

In [31]:
# Собираем сводную таблицу
# Строки — модели (popularity, content_based, item_based_cf, als)
# Столбцы — метрики из evaluate
# Используем значения, сохранённые в metrics_pop, metrics_content, metrics_item_cf, metrics_als
summary = pd.DataFrame([metrics_pop, metrics_content, metrics_item_cf, metrics_als], index=["Popularity", "Content-based", "Item-based CF", "ALS"])
summary.columns = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10", "Coverage"]

display(summary.round(4))

,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage
Popularity,0.0282,0.0653,0.0213,0.0463,0.028
Content-based,0.0201,0.0557,0.0185,0.0394,0.966
Item-based CF,0.1008,0.2769,0.1446,0.2208,0.880
ALS,0.0791,0.2214,0.1239,0.1865,0.962


## Итоговые выводы

Лучший результат по основной метрике у item-based CF: NDCG@10 = 0,2208. Это в 4,8 раза выше popularity baseline (0,0463) и выше ALS (0,1865) и content-based (0,0394). Item-based CF и ALS используют паттерны поведения, content-based - содержание книг, а popularity - глобальное число событий; в этих данных поведенческий сигнал оказался наиболее полезен для ранжирования.

Item-based CF и ALS обошли baseline по всем четырём метрикам точности; content-based не обошла его ни по одной. Самое высокое значение Precision@10 среди моделей - 0,1008, поэтому попадания в короткую десятку остаются сложной задачей. Coverage показывает компромисс: content-based покрывает 0,966 каталога, но имеет NDCG@10 0,0394; у лучшей по NDCG модели coverage ниже - 0,880. ALS сочетает coverage 0,962 и NDCG@10 0,1865, но не достигает результата item-based CF.

Для ReadMe стоит взять в продакшен item-based CF: она показывает лучший NDCG@10, существенно обгоняет текущий popularity baseline и при каталоге из 500 книг может скорить весь каталог без отдельного этапа кандидатов. Офлайн-оценка проверяет прошлые оценки и не измеряет фактическую конверсию главной страницы; перед окончательным внедрением нужен A/B-тест с метриками продуктового поведения.